# Gestión de logs de Moodle
- Marcelo Verteramo Pérsico
- https://github.com/verteramo/ubumonitorweb

In [92]:
import re, yaml, pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

### Vista de eventos agrupados por componente y ejemplos de la descripción de cada uno

In [93]:
logs = pd.read_json("logs.json")
logs.groupby(['component', 'eventname'])['description'].first().reset_index()

,component,eventname,description
0,Assignment,A submission has been submitted.,The user with id '44' has submitted the submission with id '150' for the assignment with course module id '1187'.
1,Assignment,Course module instance list viewed,The user with id '3' viewed the instance list for the module 'assign' in the course with id '72'.
2,Assignment,Course module viewed,The user with id '3' viewed the 'assign' activity with course module id '1187'.
3,Assignment,Grading form viewed,The user with id '3' viewed the grading form for the user with id '45' for the assignment with course module id '1187'.
4,Assignment,Grading table viewed,The user with id '20' viewed the grading table for the assignment with course module id '1181'.
5,Assignment,Submission confirmation form viewed.,The user with id '44' viewed the submission confirmation form for the assignment with course module id '1187'.
6,Assignment,Submission form viewed.,The user with id '49' viewed their submission for the assignment with course module id '1187'.
7,Assignment,The allocated marker has been updated.,The user with id '3' has set the marker for the user with id '46' to '10' for the assignment with course module id '1181'.
8,Assignment,The status of the submission has been viewed.,The user with id '3' has viewed the submission status page for the assignment with course module id '1187'.
9,Assignment,The submission has been graded.,The user with id '3' has graded the submission '77' for the user with id '46' for the assignment with course module id '1181'.


### Inferencia de identificadores en la descripción

In [ ]:
# Infiere identificadores dentro del texto de una descripción; Por ejemplo:
# `The user with id '20' subscribed the user with id '20' to the discussion  with id '214' in the forum with the course module id '1169'.`
# Campos en orden de aparición: userId, targetUserId, discussionId, moduleId.
#
# De manera que, extrayendo los números enteros en orden y emparejándolos con sus campos se tiene:
# Campos:  userId targetUserId discussionId moduleId
# Valores: 20     20           214          1169
#
def infer_fields(description):
    # Mapa de expresiones regulares y su identificador asociado
    patterns_map = [
        (r"user with id", "userId"),
        (r"course module(?: with)? id", "moduleId"),
        (r"course with (?:the )?id", "courseId"),
        (r"submission (?:with id )?", "submissionId"),
        (r"chapter with id", "chapterId"),
        (r"section with id", "sectionId"),
        (r"discussion (?:with id )?", "discussionId"),
        (r"post with id", "postId"),
        (r"grade item with id", "gradeItemId"),
        (r"event .*? with id", "eventId"),
        (r"glossary entry with id", "glossaryEntryId"),
        (r"role with id", "roleId"),
        (r"H5P with the id", "h5pId"),
        (r"comment with id", "commentId"),
        (r"attempt with id", "attemptId"),
        (r"grade with id", "gradeId"),
        (r"group with id", "groupId"),
        (r"tag with id", "tagId"),
        (r"enrolment method .*? with id", "enrolmentId"),
    ]

    matches = []
    for pattern, field in patterns_map:
        for match in re.finditer(pattern, description, re.IGNORECASE):
            matches.append((match.start(), field))

    # Las tuplas son del tipo: (posición, identificador),
    # por lo que, al ordenarlas por posición, valga la redundancia, los
    # identificadores se guardan en el orden de aparición en la descripción
    matches.sort(key=lambda x: x[0])
    found_fields = [match[1] for match in matches]

    # Si el campo de usuario aparece por segunda vez,
    # el usuario es el objetivo de la acción (targetUserId)
    user_count = 0
    final_fields = []
    for field in found_fields:
        if field == "userId":
            user_count += 1
            final_fields.append("targetUserId" if user_count > 1 else "userId")
        else:
            final_fields.append(field)

    return final_fields

### Generación de la configuración en YAML

In [96]:
# Se descartan eventos duplicados
unique_logs = logs[["component", "eventname", "description"]].drop_duplicates(
    subset=["component", "eventname"]
)

# Construcción del mapa de eventos con sus identificadores,
# agrupados por componente
mappings = {}
for _, row in unique_logs.iterrows():
    component = row["component"]

    fields = infer_fields(row["description"])

    if component not in mappings:
        mappings[component] = {}

    mappings[component][row["eventname"]] = fields

# Impresión de la configuración para el backend.
#
# flow_style=None
# Se evita el flow_style=True porque hace YAML más verboso, utilizando llaves y comas, asemejándose a JSON;
# Tampoco se descarta por completo (flow_style=False) porque se prefieren las listas inline [...].
#
# sort_keys=True
# No es estrictamente necesario, pero se mantienen los componentes y eventos ordenados
#
# allow_unicode=True
# Tampoco es estrictamente necesario ya que se trabaja sobre strings en language=en,
# además, independientemente del idioma, los identificadores mantienen su posición,
# pero se conserva la compatibilidad con Unicode para evitar imprevistos.
#
print(yaml.dump(mappings, default_flow_style=None, sort_keys=True, allow_unicode=True))

Assignment:
  A submission has been submitted.: [userId, submissionId, moduleId]
  Course module instance list viewed: [userId, courseId]
  Course module viewed: [userId, moduleId]
  Grading form viewed: [userId, targetUserId, moduleId]
  Grading table viewed: [userId, moduleId]
  Submission confirmation form viewed.: [userId, submissionId, moduleId]
  Submission form viewed.: [userId, submissionId, moduleId]
  The allocated marker has been updated.: [userId, targetUserId, moduleId]
  The status of the submission has been viewed.: [userId, submissionId, moduleId]
  The submission has been graded.: [userId, submissionId, targetUserId, moduleId]
  The user has accepted the statement of the submission.: [userId, submissionId, moduleId]
Book:
  Chapter viewed: [userId, chapterId, moduleId]
  Course module viewed: [userId, moduleId]
Choice:
  Choice answer added: [userId, targetUserId, moduleId]
  Course module viewed: [userId, moduleId]
File submissions:
  A file has been uploaded.: [userI